## Fake News Classifier Using LSTM

Dataset: https://www.kaggle.com/datasets/ronikdedhia/fake-news


1.dataset
2.independent and dependent features
3.cleaning the date
  1.Stemming 2.stopwords
4.Fix a sentence length to fix the input
5.One hot representation,Embedding layer
6.LSTM neural network

In [74]:
import pandas as pd
import numpy as np

In [75]:
df = pd.read_csv(
    'fake_news.csv',
    engine='python',
    on_bad_lines='skip'
)

In [76]:
df.head()

,id,title,author,text,label
0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...,1
1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...,0
2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ...",1
3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...,1
4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...,1


In [77]:
df.shape

(20822, 5)

In [78]:
df.head()

,id,title,author,text,label
0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...,1
1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...,0
2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ...",1
3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...,1
4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...,1


In [79]:
df.isnull().sum()

,0
id,0
title,565
author,1975
text,59
label,22


In [80]:
###Drop Nan Values
df=df.dropna()


In [81]:
df.head()

,id,title,author,text,label
0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...,1
1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...,0
2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ...",1
3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...,1
4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...,1


In [84]:
df = df[df['label'].astype(str).isin(['0', '1'])].copy()

In [85]:
## Get the Independent Features

X=df.drop('label',axis=1)

In [86]:
## Get the Dependent features
y=df['label']
y=np.array(y).astype('float32')
print(y.dtype)
print(y[:10])

float32
[1. 0. 1. 1. 1. 0. 0. 0. 0. 0.]


In [87]:
X.shape

(18283, 4)

In [88]:
y.shape

(18283,)

In [89]:
import tensorflow as tf

In [90]:
tf.__version__

'2.20.0'

In [91]:
from tensorflow.keras.layers import Embedding,LSTM,Dense,Dropout,Input
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import one_hot
# from tensorflow.keras.layers import LSTM
# from tensorflow.keras.layers import Dense

In [92]:
### Vocabulary size
voc_size=5000

### Onehot Representation

In [93]:
messages=X.copy()

In [94]:
messages['title'][1]

'FLYNN: Hillary Clinton, Big Woman on Campus - Breitbart'

In [95]:
messages

,id,title,author,text
0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...
1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...
2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ..."
3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...
4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...
...,...,...,...,...
20817,20795,Rapper T.I.: Trump a ’Poster Child For White S...,Jerome Hudson,Rapper T. I. unloaded on black celebrities who...
20818,20796,"N.F.L. Playoffs: Schedule, Matchups and Odds -...",Benjamin Hoffman,When the Green Bay Packers lost to the Washing...
20819,20797,Macy’s Is Said to Receive Takeover Approach by...,Michael J. de la Merced and Rachel Abrams,The Macy’s of today grew from the union of sev...
20820,20798,"NATO, Russia To Hold Parallel Exercises In Bal...",Alex Ansary,"NATO, Russia To Hold Parallel Exercises In Bal..."


In [97]:
# messages.reset_index(inplace=True)

In [98]:
messages

,index,id,title,author,text
0,0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...
1,1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...
2,2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ..."
3,3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...
4,4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...
...,...,...,...,...,...
18278,20817,20795,Rapper T.I.: Trump a ’Poster Child For White S...,Jerome Hudson,Rapper T. I. unloaded on black celebrities who...
18279,20818,20796,"N.F.L. Playoffs: Schedule, Matchups and Odds -...",Benjamin Hoffman,When the Green Bay Packers lost to the Washing...
18280,20819,20797,Macy’s Is Said to Receive Takeover Approach by...,Michael J. de la Merced and Rachel Abrams,The Macy’s of today grew from the union of sev...
18281,20820,20798,"NATO, Russia To Hold Parallel Exercises In Bal...",Alex Ansary,"NATO, Russia To Hold Parallel Exercises In Bal..."


In [99]:
import nltk
import re
from nltk.corpus import stopwords

In [100]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [101]:
### Dataset Preprocessing
from nltk.stem.porter import PorterStemmer ##stemming purpose
ps = PorterStemmer()
corpus = []
for i in range(0, len(messages)):
    review = re.sub('[^a-zA-Z]', ' ', messages['title'][i])
    review = review.lower()
    review = review.split()

    review = [ps.stem(word) for word in review if not word in stopwords.words('english')]
    # lematization take more time athor wish i can use lematization
    review = ' '.join(review)
    corpus.append(review)

In [102]:
corpus

['hous dem aid even see comey letter jason chaffetz tweet',
 'flynn hillari clinton big woman campu breitbart',
 'truth might get fire',
 'civilian kill singl us airstrik identifi',
 'iranian woman jail fiction unpublish stori woman stone death adulteri',
 'jacki mason hollywood would love trump bomb north korea lack tran bathroom exclus video breitbart',
 'beno hamon win french socialist parti presidenti nomin new york time',
 'back channel plan ukrain russia courtesi trump associ new york time',
 'obama organ action partner soro link indivis disrupt trump agenda',
 'bbc comedi sketch real housew isi caus outrag',
 'russian research discov secret nazi militari base treasur hunter arctic photo',
 'us offici see link trump russia',
 'ye paid govern troll social media blog forum websit',
 'major leagu soccer argentin find home success new york time',
 'well fargo chief abruptli step new york time',
 'anonym donor pay million releas everyon arrest dakota access pipelin',
 'fbi close hilla

In [103]:
corpus[1]

'flynn hillari clinton big woman campu breitbart'

In [104]:
onehot_repr=[one_hot(words,voc_size)for words in corpus]
onehot_repr

[[2447, 324, 3290, 3944, 1583, 3299, 4405, 3292, 4992, 4785],
 [3602, 1462, 4847, 3272, 2974, 3239, 4160],
 [1470, 1264, 3840, 229],
 [190, 555, 4366, 2365, 4109, 2201],
 [864, 2974, 2589, 3660, 1208, 1359, 2974, 4415, 719, 2571],
 [3474,
  988,
  1680,
  4040,
  907,
  4785,
  1144,
  1305,
  2830,
  909,
  929,
  4283,
  4372,
  3618,
  4160],
 [4237, 2557, 1137, 1266, 2936, 2953, 4083, 286, 3307, 530, 3541],
 [1187, 3408, 4251, 4558, 3988, 837, 4785, 1443, 3307, 530, 3541],
 [2655, 1401, 2737, 4568, 4613, 2343, 1580, 1665, 4785, 3489],
 [1423, 2964, 4111, 3394, 4691, 3208, 2277, 374],
 [3178, 3286, 1509, 3703, 3428, 4965, 3320, 433, 144, 2850, 2758],
 [2365, 4086, 1583, 2343, 4785, 3988],
 [2620, 2708, 545, 2032, 4159, 3944, 4840, 3198, 911],
 [667, 2969, 4604, 1670, 3592, 376, 4026, 3307, 530, 3541],
 [2397, 1024, 382, 4022, 3416, 3307, 530, 3541],
 [830, 2932, 393, 4903, 3262, 17, 1174, 1393, 2244, 398],
 [154, 2962, 1462],
 [4499, 3782, 1043, 2666, 4785, 4411, 2610, 4160],
 [4129

In [105]:
corpus[1]

'flynn hillari clinton big woman campu breitbart'

In [106]:
onehot_repr[1]

[3602, 1462, 4847, 3272, 2974, 3239, 4160]

### Embedding Representation

In [107]:
sent_length=20
embedded_docs=pad_sequences(onehot_repr,padding='post',maxlen=sent_length)
print(embedded_docs)

[[2447  324 3290 ...    0    0    0]
 [3602 1462 4847 ...    0    0    0]
 [1470 1264 3840 ...    0    0    0]
 ...
 [1267 3413 1558 ...    0    0    0]
 [3621 3988   11 ...    0    0    0]
 [1592    6 2306 ...    0    0    0]]


In [108]:
embedded_docs[1]

array([3602, 1462, 4847, 3272, 2974, 3239, 4160,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0], dtype=int32)

In [109]:
embedded_docs[0]

array([2447,  324, 3290, 3944, 1583, 3299, 4405, 3292, 4992, 4785,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0], dtype=int32)

In [110]:
## Creating model
embedding_vector_features=40 ##features representation
model = Sequential([
    Input(shape=(sent_length,)),
    Embedding(voc_size, embedding_vector_features),
    LSTM(100),
    Dense(1, activation='sigmoid')
])
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])
print(model.summary())

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 20, 40)         │       200,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 100)            │        56,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │           101 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 256,501 (1001.96 KB)

 Trainable params: 256,501 (1001.96 KB)

 Non-trainable params: 0 (0.00 B)

None


In [111]:
len(embedded_docs),y.shape

(18283, (18283,))

In [112]:
import numpy as np
X_final=np.array(embedded_docs)
y_final=np.array(y)

In [113]:
X_final.shape,y_final.shape

((18283, 20), (18283,))

In [114]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_final, y_final, test_size=0.33, random_state=42)

### Model Training

In [124]:
### Finally Training
model.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=10,batch_size=64)

Epoch 1/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.8546 - loss: 0.3054 - val_accuracy: 0.9142 - val_loss: 0.2083
Epoch 2/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.9397 - loss: 0.1573 - val_accuracy: 0.9128 - val_loss: 0.2224
Epoch 3/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9571 - loss: 0.1177 - val_accuracy: 0.9084 - val_loss: 0.2522
Epoch 4/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9651 - loss: 0.0924 - val_accuracy: 0.9049 - val_loss: 0.3074
Epoch 5/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9718 - loss: 0.0778 - val_accuracy: 0.9112 - val_loss: 0.2937
Epoch 6/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9776 - loss: 0.0647 - val_accuracy: 0.9021 - val_loss: 0.2824
Epoch 7/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.9775 - loss: 0.0611 - val_accuracy: 0.9044 - val_loss: 0.3376
Epoch 8/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.9847 - loss: 0.0461 - val_accuracy: 

### Adding Dropout

In [123]:
from tensorflow.keras.layers import Dropout
## Creating model
embedding_vector_features=40
model = Sequential([
    Input(shape=(sent_length,)),
    Embedding(voc_size, embedding_vector_features),
    Dropout(0.3),
    LSTM(100),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])
print(model.summary())

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, 20, 40)         │       200,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 20, 40)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 100)            │        56,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │           101 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 256,501 (1001.96 KB)

 Trainable params: 256,501 (1001.96 KB)

 Non-trainable params: 0 (0.00 B)

None


### Performance Metrics And Accuracy

In [125]:
y_pred=model.predict(X_test)

189/189 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step


In [126]:
y_pred=np.where(y_pred > 0.6, 1,0) ##AUC ROC Curve

In [127]:
from sklearn.metrics import confusion_matrix

In [128]:
confusion_matrix(y_test,y_pred)

array([[3121,  294],
       [ 295, 2324]])

In [129]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.9023864766324163

In [130]:
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

         0.0       0.91      0.91      0.91      3415
         1.0       0.89      0.89      0.89      2619

    accuracy                           0.90      6034
   macro avg       0.90      0.90      0.90      6034
weighted avg       0.90      0.90      0.90      6034

